In [3]:
from IPython.display import Audio, display
import soundfile as sf
import wave
from pydub import AudioSegment, utils
import librosa
import torchaudio
import numpy as np
from pathlib import Path
import tempfile

# Generated audio goes to a temp folder outside the repo
OUT_DIR = Path(tempfile.gettempdir()) / "ml-notebooks" / "audio-manipulation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named '_cffi_backend'

In [ ]:
# Read Audio using IPython
sample = "sample.wav"
display(Audio(sample))

In [ ]:
# Read and write audio using soundfile
sample = "sample.wav"
data, sample_rate = sf.read(sample)
sf.write(OUT_DIR / "sample_mp3.mp3", data, sample_rate)

print(data)

[[-0.12939453 -0.11911011]
 [-0.17141724 -0.17520142]
 [-0.14825439 -0.16662598]
 ...
 [ 0.          0.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]]


In [ ]:
# Read audio detail
sample = "sample.wav"
object = wave.open(sample, "r")
print("Number of channels", object.getnchannels())
print("Sample width: ", object.getsampwidth())
print("Frame rate: ", object.getframerate())
print("Number of frame: ", object.getnframes())

# Write audio
frames = object.readframes(object.getnframes())
object.close()

object = wave.open(str(OUT_DIR / "sample_new.wav"), "w")
object.setnchannels(1)
object.setsampwidth(3)
object.setframerate(88000)
object.writeframes(frames)
object.close()

Number of channels 2
Sample width:  2
Frame rate:  44100
Number of frame:  654383


AudioSegment -> + - decible, loop, fade in & out, speedup, reverse.

In [ ]:
# Audio Augmentation
# Audio Segment
sample = "sample.wav"
audio = AudioSegment.from_wav(sample)

# Cut audio
# [Cut di awal : Cut di akhir]
audio = audio[:3000]  # Cut di 3 detik pertama dan cut di 3 detik terakhir
audio.export(str(OUT_DIR / "sample_cut.wav"), format='wav')

# Menambahkan atau mengurangi decible
audio = AudioSegment.from_wav(sample)
audio = audio + 200 # Tambah decible
audio.export(str(OUT_DIR / "sample_big.wav"), format='wav')
audio = audio - 200 # Kurangi decible

audio = AudioSegment.from_wav(sample)
audio = audio - 200 # Kurangi decible
audio.export(str(OUT_DIR / "sample_small.wav"), format='wav')

# Membuat clip berulang sebanyak n kali
audio = AudioSegment.from_wav(sample)
audio = audio * 10              # clip 10 kali
audio.export(str(OUT_DIR / "sample_ulang.wav"), format='wav')
audio = audio + audio + audio   # clip 3 kali

# Efek fade in
audio = AudioSegment.from_wav(sample)
audio = audio.fade_in(3000)     # Fade in di 3 detik awal
audio.export(str(OUT_DIR / "sample_fadein.wav"), format='wav')

# Efek fade out
audio = AudioSegment.from_wav(sample)
audio = audio.fade_out(3000)    # Fade out di 3 detik akhir
audio.export(str(OUT_DIR / "sample_fadeout.wav"), format='wav')

# Efek fade in dan fade out bersamaan
audio = audio.fade_in(3000).fade_out(3000)

# Efek sped up
audio = AudioSegment.from_wav(sample)
audio = audio.speedup(2)
audio.export(str(OUT_DIR / "sample_spedup.wav"), format='wav')

# Efek reverse
audio = AudioSegment.from_wav(sample)
audio = audio.reverse()
audio.export(str(OUT_DIR / "sample_reverse.wav"), format='wav')

<_io.BufferedRandom name='sample_reverse.wav'>

Librosa -> white noise, time stretch, pitch shift, random gain

In [ ]:
# Librosa
# white_noise
def white_noise(signal, noise_factor):
    noise = np.random.normal(0, signal.std(), signal.size)
    augmented_audio = signal + noise * noise_factor
    return augmented_audio

signal, sr = librosa.load(sample, sr=44100)
augmented_audio = white_noise(signal, 0.3)
sf.write(OUT_DIR / "sample_white_noise.wav", augmented_audio, sr)

# time stretch
def time_stretch(signal, time_stretch_rate):
    return librosa.effects.time_stretch(signal, rate=time_stretch_rate)

signal, sr = librosa.load(sample, sr=44100)
augmented_audio = time_stretch(signal, 3)   # mempercepat
sf.write(OUT_DIR / "sample_speedup.wav", augmented_audio, sr)
augmented_audio = time_stretch(signal, 0.3) # memperlambat
sf.write(OUT_DIR / "sample_speeddown.wav", augmented_audio, sr)

# pitch shift
def pitch_shift(signal, sr, num_step):
    return librosa.effects.pitch_shift(signal, sr=sr, n_steps=num_step)

signal, sr = librosa.load(sample, sr=44100)
augmented_audio = pitch_shift(signal, sr, 12)
sf.write(OUT_DIR / "sample_pitch_shift.wav", augmented_audio, sr)

# random gain -> memperbesar atau memperkecil decible
def random_gain(signal, min_gain_factor, max_gain_factor):
    gain_factor = np.random.uniform(min_gain_factor, max_gain_factor)
    augmented_audio = signal + gain_factor
    return augmented_audio

signal, sr = librosa.load(sample, sr=44100)
augmented_audio = random_gain(signal, 0.1, 3)
sf.write(OUT_DIR / "sample_random.wav", augmented_audio, sr)